# Set up

In [ ]:
import pandas as pd
import spacy
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import re
import string

# Read the data

In [ ]:
#read the data
df = pd.read_csv('/content/SAGEUSA_clean_data.csv')
df

,id,post_owner.username,text,clean_text
0,6.147890e+14,SAGEUSA,"🌈🧠 This #MentalHealthAwarenessMonth, SAGE is p...","this , sage is proud to feature our partnersh..."
1,1.553641e+15,SAGEUSA,💡A study on suicidal behavior and coming out m...,a study on suicidal behavior and coming out mi...
2,1.356778e+15,SAGEUSA,REMINDER‼️🏳️‍🌈 Celebrate Pride Month at SAGE &...,reminder‼‍ celebrate pride month at sage & fri...
3,1.047445e+15,SAGEUSA,"📣 SAGE thanks Congressman Robert Garcia, Repre...","sage thanks congressman robert garcia, repres..."
4,1.889815e+15,SAGEUSA,📣 Minnesota is making history! 🏛️💙 SAGE commen...,minnesota is making history! sage commends r...
...,...,...,...,...
5739,1.069095e+15,SAGEUSA,Check out the link for a variety of informativ...,check out the link for a variety of informativ...
5740,7.236230e+14,SAGEUSA,Thank you to all who participated in our Inter...,thank you to all who participated in our inter...
5741,1.206087e+15,SAGEUSA,On The Huffington Post too!,on the huffington post too!
5742,9.490401e+15,SAGEUSA,Don't forget our SAGE Harlem Thanksgiving Potl...,don't forget our sage harlem thanksgiving potl...


In [ ]:
df.columns

Index(['id', 'post_owner.username', 'text', 'clean_text'], dtype='object')

# Sentiment analysis with VADER

In [ ]:
!pip install vaderSentiment

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

In [ ]:
# define a function to get sentiment scores and classify sentiment
def analyze_VADER_sentiment(text):
    # Check if the input is a string; if not, convert it to a string.
    if not isinstance(text, str):
        text = str(text)

    scores = analyzer.polarity_scores(text)

    # Classify sentiment based on the compound score
    if scores['compound'] >= 0.05:
        sentiment = 'Positive'
    elif scores['compound'] <= -0.05:
        sentiment = 'Negative'
    else:
        sentiment = 'Neutral'

    return pd.Series([scores['neg'], scores['neu'], scores['pos'], scores['compound'], sentiment])

In [ ]:
# Apply the sentiment analysis function to the 'clean_text' column
df[['VADER_neg', 'VADER_neu', 'VADER_pos', 'VADER_compound', 'VADER_sentiment']] = df['clean_text'].apply(analyze_VADER_sentiment)
df

,id,post_owner.username,text,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment
0,6.147890e+14,SAGEUSA,"🌈🧠 This #MentalHealthAwarenessMonth, SAGE is p...","this , sage is proud to feature our partnersh...",0.000,0.902,0.098,0.8172,Positive
1,1.553641e+15,SAGEUSA,💡A study on suicidal behavior and coming out m...,a study on suicidal behavior and coming out mi...,0.149,0.814,0.037,-0.8519,Negative
2,1.356778e+15,SAGEUSA,REMINDER‼️🏳️‍🌈 Celebrate Pride Month at SAGE &...,reminder‼‍ celebrate pride month at sage & fri...,0.026,0.643,0.330,0.9609,Positive
3,1.047445e+15,SAGEUSA,"📣 SAGE thanks Congressman Robert Garcia, Repre...","sage thanks congressman robert garcia, repres...",0.000,0.833,0.167,0.8402,Positive
4,1.889815e+15,SAGEUSA,📣 Minnesota is making history! 🏛️💙 SAGE commen...,minnesota is making history! sage commends r...,0.000,0.838,0.162,0.9359,Positive
...,...,...,...,...,...,...,...,...,...
5739,1.069095e+15,SAGEUSA,Check out the link for a variety of informativ...,check out the link for a variety of informativ...,0.000,1.000,0.000,0.0000,Neutral
5740,7.236230e+14,SAGEUSA,Thank you to all who participated in our Inter...,thank you to all who participated in our inter...,0.000,0.823,0.177,0.6808,Positive
5741,1.206087e+15,SAGEUSA,On The Huffington Post too!,on the huffington post too!,0.000,1.000,0.000,0.0000,Neutral
5742,9.490401e+15,SAGEUSA,Don't forget our SAGE Harlem Thanksgiving Potl...,don't forget our sage harlem thanksgiving potl...,0.000,0.734,0.266,0.3699,Positive


#Syntactic Complexity_Analysis

In [ ]:
#Load the spaCy model
nlp = spacy.load("en_core_web_sm")


In [ ]:
def spacy_TTR(text):

    if not isinstance(text, str):
        return pd.Series([0, 0, 0, 0.0])  # Handle non-string inputs gracefully

    # Process the text using spaCy
    doc = nlp(text)

    # Calculate sentence count
    sent_count = len(list(doc.sents)) #doc.sents:SpaCy uses its sentence boundary detection (SBD) algorithm to split the text into sentences.

    # Extract words (excluding punctuation and spaces)
    words = [token.text.lower() for token in doc if token.is_alpha]
    word_count = len(words)

    # Calculate unique word count
    unique_word_count = len(set(words))

    # Calculate Type-Token Ratio (TTR)
    TTR = unique_word_count / word_count if word_count > 0 else 0

    # Return the results as a pandas Series
    return pd.Series([sent_count, word_count, unique_word_count, TTR])

In [ ]:
def syntactic_complexity_metrics(text):

    if not isinstance(text, str):
        return pd.Series([0, 0, 0, 0, 0, 0])  # Handle non-string inputs gracefully

    # Process the text using spaCy
    doc = nlp(text)

    sen_count = 0     # Total number of sentences (treated as clauses)
    total_dependents = 0
    total_coordinate_phrases = 0
    total_complex_nominals = 0

    # Iterate over sentences in the text
    for sent in doc.sents:
        sen_count += 1

        # Initialize counters for the current sentence
        sent_dependents = 0
        sent_coord_phrases = 0
        sent_complex_nominals = 0

        for token in sent:
            # 1. Count dependents per clause
            if token.dep_ in {'advcl', 'csubj', 'ccomp', 'acl', 'xcomp', 'relcl'}:
                sent_dependents += 1

            # 2. Count coordinate phrases per clause
            if token.dep_ in {'cc', 'conj'}:
                sent_coord_phrases += 1

            # 3. Count complex nominals per clause
            if token.dep_ in {"nsubj", "dobj", "pobj", "iobj", "nmod"}:
                if any(child.dep_ in {"amod", "poss", "compound"} for child in token.children):
                    sent_complex_nominals += 1

        total_dependents += sent_dependents
        total_coordinate_phrases += sent_coord_phrases
        total_complex_nominals += sent_complex_nominals

    # Avoid division by zero (if no sentences are found)
    if sen_count == 0:
        return pd.Series([0, 0, 0, 0, 0, 0])

    # Calculate per-clause averages
    dependents_per_clause = total_dependents / sen_count
    coord_phrases_per_clause = total_coordinate_phrases / sen_count
    complex_nominals_per_clause = total_complex_nominals / sen_count

    # Return results as a pandas Series
    return pd.Series([
        total_dependents, dependents_per_clause,
        total_coordinate_phrases, coord_phrases_per_clause,
        total_complex_nominals, complex_nominals_per_clause
    ])

In [ ]:
# apply those two functions to our dataset
df[['sent_count', 'word_count', 'unique_word_count', 'TTR']] = df['clean_text'].apply(spacy_TTR)

df[['total_dependents', 'dependents_per_clause',
    'total_coord_phrases', 'coord_phrases_per_clause',
    'total_complex_nominals', 'complex_nominals_per_clause']] = df['clean_text'].apply(syntactic_complexity_metrics)

df

,id,post_owner.username,text,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause
0,6.147890e+14,SAGEUSA,"🌈🧠 This #MentalHealthAwarenessMonth, SAGE is p...","this , sage is proud to feature our partnersh...",0.000,0.902,0.098,0.8172,Positive,4.0,65.0,54.0,0.830769,4.0,1.000000,7.0,1.750000,8.0,2.000000
1,1.553641e+15,SAGEUSA,💡A study on suicidal behavior and coming out m...,a study on suicidal behavior and coming out mi...,0.149,0.814,0.037,-0.8519,Negative,3.0,57.0,52.0,0.912281,4.0,1.333333,9.0,3.000000,8.0,2.666667
2,1.356778e+15,SAGEUSA,REMINDER‼️🏳️‍🌈 Celebrate Pride Month at SAGE &...,reminder‼‍ celebrate pride month at sage & fri...,0.026,0.643,0.330,0.9609,Positive,3.0,37.0,36.0,0.972973,2.0,0.666667,10.0,3.333333,5.0,1.666667
3,1.047445e+15,SAGEUSA,"📣 SAGE thanks Congressman Robert Garcia, Repre...","sage thanks congressman robert garcia, repres...",0.000,0.833,0.167,0.8402,Positive,1.0,43.0,37.0,0.860465,2.0,2.000000,9.0,9.000000,2.0,2.000000
4,1.889815e+15,SAGEUSA,📣 Minnesota is making history! 🏛️💙 SAGE commen...,minnesota is making history! sage commends r...,0.000,0.838,0.162,0.9359,Positive,5.0,83.0,64.0,0.771084,8.0,1.600000,12.0,2.400000,5.0,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5739,1.069095e+15,SAGEUSA,Check out the link for a variety of informativ...,check out the link for a variety of informativ...,0.000,1.000,0.000,0.0000,Neutral,1.0,18.0,16.0,0.888889,0.0,0.000000,3.0,3.000000,1.0,1.000000
5740,7.236230e+14,SAGEUSA,Thank you to all who participated in our Inter...,thank you to all who participated in our inter...,0.000,0.823,0.177,0.6808,Positive,2.0,28.0,26.0,0.928571,2.0,1.000000,4.0,2.000000,1.0,0.500000
5741,1.206087e+15,SAGEUSA,On The Huffington Post too!,on the huffington post too!,0.000,1.000,0.000,0.0000,Neutral,1.0,5.0,5.0,1.000000,0.0,0.000000,0.0,0.000000,1.0,1.000000
5742,9.490401e+15,SAGEUSA,Don't forget our SAGE Harlem Thanksgiving Potl...,don't forget our sage harlem thanksgiving potl...,0.000,0.734,0.266,0.3699,Positive,1.0,8.0,8.0,1.000000,0.0,0.000000,0.0,0.000000,1.0,1.000000


# Cosine similarity

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer, util

In [ ]:
model = SentenceTransformer("all-mpnet-base-v2")

In [ ]:
#Processes text data to compute the average similarity for each row in a DataFrame.
def process_text_average_similarity(data, text_column='content_clean'):
    # Ensure data is a DataFrame
    if not isinstance(data, pd.DataFrame):
        data = pd.DataFrame(data)

    # Step 1: Split text into sentences
    def split_into_sentences(text):
        return [sent.text.strip() for sent in nlp(text).sents]

    data['sent_list'] = data[text_column].apply(split_into_sentences)

    # Step 2: Encode sentences into embeddings
    model = SentenceTransformer("all-mpnet-base-v2")
    data['sentence_embeddings'] = data['sent_list'].apply(
        lambda sentences: [model.encode(sentence, convert_to_tensor=True) for sentence in sentences]
    )

    # Step 3: Compute pairwise cosine similarities
    def compute_pairwise_similarity(embeddings):
        if len(embeddings) < 2:
            return None  # Return None if fewer than 2 sentences
        similarities = []
        for i in range(len(embeddings) - 1):
            sim = util.pytorch_cos_sim(embeddings[i], embeddings[i + 1]).item()
            similarities.append(sim)
        return similarities

    data['pair_similarity'] = data['sentence_embeddings'].apply(compute_pairwise_similarity)

    # Step 4: Calculate the average similarity
    data['average_similarity'] = data['pair_similarity'].apply(
        lambda sims: np.mean(sims) if sims else None
    )

    # Drop intermediate columns to keep only the average similarity
    data = data.drop(columns=['sent_list', 'sentence_embeddings', 'pair_similarity'])

    return data

In [ ]:
# Apply the function and add the average similarity as a new column
df = process_text_average_similarity(df, text_column='clean_text')
df

,id,post_owner.username,text,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,word_count,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause,average_similarity
0,6.147890e+14,SAGEUSA,"🌈🧠 This #MentalHealthAwarenessMonth, SAGE is p...","this , sage is proud to feature our partnersh...",0.000,0.902,0.098,0.8172,Positive,4.0,65.0,54.0,0.830769,4.0,1.000000,7.0,1.750000,8.0,2.000000,0.467838
1,1.553641e+15,SAGEUSA,💡A study on suicidal behavior and coming out m...,a study on suicidal behavior and coming out mi...,0.149,0.814,0.037,-0.8519,Negative,3.0,57.0,52.0,0.912281,4.0,1.333333,9.0,3.000000,8.0,2.666667,0.216879
2,1.356778e+15,SAGEUSA,REMINDER‼️🏳️‍🌈 Celebrate Pride Month at SAGE &...,reminder‼‍ celebrate pride month at sage & fri...,0.026,0.643,0.330,0.9609,Positive,3.0,37.0,36.0,0.972973,2.0,0.666667,10.0,3.333333,5.0,1.666667,0.397427
3,1.047445e+15,SAGEUSA,"📣 SAGE thanks Congressman Robert Garcia, Repre...","sage thanks congressman robert garcia, repres...",0.000,0.833,0.167,0.8402,Positive,1.0,43.0,37.0,0.860465,2.0,2.000000,9.0,9.000000,2.0,2.000000,NaN
4,1.889815e+15,SAGEUSA,📣 Minnesota is making history! 🏛️💙 SAGE commen...,minnesota is making history! sage commends r...,0.000,0.838,0.162,0.9359,Positive,5.0,83.0,64.0,0.771084,8.0,1.600000,12.0,2.400000,5.0,1.000000,0.356201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5739,1.069095e+15,SAGEUSA,Check out the link for a variety of informativ...,check out the link for a variety of informativ...,0.000,1.000,0.000,0.0000,Neutral,1.0,18.0,16.0,0.888889,0.0,0.000000,3.0,3.000000,1.0,1.000000,NaN
5740,7.236230e+14,SAGEUSA,Thank you to all who participated in our Inter...,thank you to all who participated in our inter...,0.000,0.823,0.177,0.6808,Positive,2.0,28.0,26.0,0.928571,2.0,1.000000,4.0,2.000000,1.0,0.500000,0.539069
5741,1.206087e+15,SAGEUSA,On The Huffington Post too!,on the huffington post too!,0.000,1.000,0.000,0.0000,Neutral,1.0,5.0,5.0,1.000000,0.0,0.000000,0.0,0.000000,1.0,1.000000,NaN
5742,9.490401e+15,SAGEUSA,Don't forget our SAGE Harlem Thanksgiving Potl...,don't forget our sage harlem thanksgiving potl...,0.000,0.734,0.266,0.3699,Positive,1.0,8.0,8.0,1.000000,0.0,0.000000,0.0,0.000000,1.0,1.000000,NaN


# Part of Speech

In [ ]:
# Tokenize the text (split each sentence into words)
df['word_list'] = df['clean_text'].astype(str).apply(lambda x: x.split())
df.head(2)

,id,post_owner.username,text,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,...,unique_word_count,TTR,total_dependents,dependents_per_clause,total_coord_phrases,coord_phrases_per_clause,total_complex_nominals,complex_nominals_per_clause,average_similarity,word_list
0,6.147890e+14,SAGEUSA,"🌈🧠 This #MentalHealthAwarenessMonth, SAGE is p...","this , sage is proud to feature our partnersh...",0.000,0.902,0.098,0.8172,Positive,4.0,...,54.0,0.830769,4.0,1.000000,7.0,1.75,8.0,2.000000,0.467838,"[this, ,, sage, is, proud, to, feature, our, p..."
1,1.553641e+15,SAGEUSA,💡A study on suicidal behavior and coming out m...,a study on suicidal behavior and coming out mi...,0.149,0.814,0.037,-0.8519,Negative,3.0,...,52.0,0.912281,4.0,1.333333,9.0,3.00,8.0,2.666667,0.216879,"[a, study, on, suicidal, behavior, and, coming..."


In [ ]:
df_subset =df[['id','clean_text','word_list']]
df_subset

,id,clean_text,word_list
0,6.147890e+14,"this , sage is proud to feature our partnersh...","[this, ,, sage, is, proud, to, feature, our, p..."
1,1.553641e+15,a study on suicidal behavior and coming out mi...,"[a, study, on, suicidal, behavior, and, coming..."
2,1.356778e+15,reminder‼‍ celebrate pride month at sage & fri...,"[reminder‼‍, celebrate, pride, month, at, sage..."
3,1.047445e+15,"sage thanks congressman robert garcia, repres...","[sage, thanks, congressman, robert, garcia,, r..."
4,1.889815e+15,minnesota is making history! sage commends r...,"[minnesota, is, making, history!, sage, commen..."
...,...,...,...
5739,1.069095e+15,check out the link for a variety of informativ...,"[check, out, the, link, for, a, variety, of, i..."
5740,7.236230e+14,thank you to all who participated in our inter...,"[thank, you, to, all, who, participated, in, o..."
5741,1.206087e+15,on the huffington post too!,"[on, the, huffington, post, too!]"
5742,9.490401e+15,don't forget our sage harlem thanksgiving potl...,"[don't, forget, our, sage, harlem, thanksgivin..."


In [ ]:
# Explode the 'word_list' column, creating a separate row for each word, and renames the exploded column to 'word'
df_exploded = df_subset.explode('word_list').rename(columns={'word_list': 'word'})
df_exploded

,id,clean_text,word
0,6.147890e+14,"this , sage is proud to feature our partnersh...",this
0,6.147890e+14,"this , sage is proud to feature our partnersh...",","
0,6.147890e+14,"this , sage is proud to feature our partnersh...",sage
0,6.147890e+14,"this , sage is proud to feature our partnersh...",is
0,6.147890e+14,"this , sage is proud to feature our partnersh...",proud
...,...,...,...
5743,3.994127e+15,"saturday, november 19, 2011 from 8 - 10 pm at ...",mad
5743,3.994127e+15,"saturday, november 19, 2011 from 8 - 10 pm at ...",stu
5743,3.994127e+15,"saturday, november 19, 2011 from 8 - 10 pm at ...","media,"
5743,3.994127e+15,"saturday, november 19, 2011 from 8 - 10 pm at ...",and


In [ ]:
#lowcasing, remove quotation marks (single quotes, double quotes, and special quotes) from the start and end of the strings
def clean_word_column(df):

    # Remove punctuation: Uses regex to remove standard punctuation (from string.punctuation) around each word.
    df['word'] = df['word'].str.replace(f"[{string.punctuation}]", "", regex=True)

    # Lowercasing: Converts each word to lowercase (we did this when preprocessing "text" column)
    df['word'] = df['word'].str.lower()

    # Quotation mark removal: Uses regex to remove single quotes, double quotes, and special quotes from the start and end of each word.
    df['word'] = df['word'].str.replace(r"^[‘’“”\"']|[‘’“”\"']$", '', regex=True)

    # Row filtering: Remove rows where 'word_lower' is NaN or contains non-string values
    df = df[df['word'].apply(lambda x: isinstance(x, str))]

    # Return the result
    return df

In [ ]:
df_lex = clean_word_column(df_exploded)
df_lex[['id','word']]

,id,word
0,6.147890e+14,this
0,6.147890e+14,
0,6.147890e+14,sage
0,6.147890e+14,is
0,6.147890e+14,proud
...,...,...
5743,3.994127e+15,mad
5743,3.994127e+15,stu
5743,3.994127e+15,media
5743,3.994127e+15,and


In [ ]:
# use Spacy: tokenize words, and POS tagging
def process_word_spacy(df, text_column='word'):
    # Ensure column is in string format and not NaN
    df = df[df[text_column].notna()]
    df[text_column] = df[text_column].astype(str)

    # Create spaCy documents using nlp.pipe for efficient processing
    docs = list(nlp.pipe(df[text_column]))

    # Initialize lists to store processing results
    token_texts = []
    lemmas = []
    pos_tags = []
    tags = []
    token_counts = []

    # Process each document and extract relevant information
    for doc in docs:
        # Token text (original form), lemmas (base form), POS (simple and detailed)
        token_texts.append([token.text for token in doc])
        lemmas.append([token.lemma_ for token in doc])
        pos_tags.append([token.pos_ for token in doc])  # Simple POS tags
        tags.append([token.tag_ for token in doc])  # Detailed POS tags
        token_counts.append(len(doc))  # Number of tokens in each doc

    # Add the results as new columns in the DataFrame
    df = df.assign(
        sp_tokenized=token_texts,
        sp_n_token=token_counts,
        sp_lemma=lemmas,
        sp_pos=pos_tags,
        sp_tag=tags)
    return df

In [ ]:
df_lex = process_word_spacy(df_lex)
df_lex

,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag
0,6.147890e+14,"this , sage is proud to feature our partnersh...",this,[this],1,[this],[PRON],[DT]
0,6.147890e+14,"this , sage is proud to feature our partnersh...",,[],0,[],[],[]
0,6.147890e+14,"this , sage is proud to feature our partnersh...",sage,[sage],1,[sage],[NOUN],[NN]
0,6.147890e+14,"this , sage is proud to feature our partnersh...",is,[is],1,[be],[AUX],[VBZ]
0,6.147890e+14,"this , sage is proud to feature our partnersh...",proud,[proud],1,[proud],[ADJ],[JJ]
...,...,...,...,...,...,...,...,...
5743,3.994127e+15,"saturday, november 19, 2011 from 8 - 10 pm at ...",mad,[mad],1,[mad],[ADJ],[JJ]
5743,3.994127e+15,"saturday, november 19, 2011 from 8 - 10 pm at ...",stu,[stu],1,[stu],[PROPN],[NNP]
5743,3.994127e+15,"saturday, november 19, 2011 from 8 - 10 pm at ...",media,[media],1,[medium],[NOUN],[NNS]
5743,3.994127e+15,"saturday, november 19, 2011 from 8 - 10 pm at ...",and,[and],1,[and],[CCONJ],[CC]


In [ ]:
# save the results to a new csv file
df_lex.to_csv('/content/SAGEUSA_results.csv', index=False)

In [ ]:
#Count the number of POS tags (group by id)
def count_pos_tags_by_speaker(df, pos_column='sp_pos', group_column='id'):

    # Ensure the specified columns exist in the DataFrame
    if pos_column not in df.columns or group_column not in df.columns:
        raise ValueError(f"Columns '{pos_column}' and '{group_column}' must exist in the DataFrame.")

    # Explode the pos_column to have one row per POS tag
    df_exploded = df.explode(pos_column)

    # Group by the specified group column and count each POS tag
    pos_counts = (
        df_exploded.groupby(group_column)[pos_column]
        .value_counts()
        .unstack(fill_value=0))  # Fill NaNs with 0s for missing POS tags

    # Add a new column for the total POS count by summing across all POS columns
    pos_counts['total_pos'] = pos_counts.sum(axis=1)

    # Return the resulting DataFrame
    return pos_counts


In [ ]:
df_pos = count_pos_tags_by_speaker(df_lex)
df_pos

sp_pos,ADJ,ADP,ADV,AUX,CCONJ,DET,INTJ,NOUN,NUM,PART,PRON,PROPN,PUNCT,SCONJ,SYM,VERB,X,total_pos
id,,,,,,,,,,,,,,,,,,
1.667270e+14,4,2,5,2,1,0,0,11,0,4,7,2,0,1,0,9,0,48
2.098540e+14,3,1,0,0,2,0,0,10,0,2,5,0,0,0,0,4,0,27
2.381550e+14,2,11,5,0,2,0,0,25,7,1,11,3,1,1,0,12,0,81
2.481730e+14,5,10,5,3,3,0,0,28,0,3,11,3,0,0,0,8,0,79
2.620290e+14,5,7,1,2,1,0,0,7,1,2,4,3,0,0,0,10,0,43
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3.032345e+16,0,3,2,0,1,0,0,15,0,2,5,1,0,0,0,5,0,34
3.052671e+16,0,3,0,0,1,0,0,2,0,1,3,0,0,0,0,4,0,14
3.071338e+16,4,4,2,3,3,0,0,12,0,2,3,1,0,2,0,7,0,43


POS: The simple UPOS part-of-speech tag

Universal POS tag: https://universaldependencies.org/u/pos/

- ADJ: adjective.
- ADP: adposition.
- ADV: adverb.
- AUX: auxiliary.
- CCONJ: coordinating conjunction.
- DET: determiner.
- INTJ: interjection.
- NOUN: noun.
- NUM: numeral.
- PART: particle.
- PRON: pronoun.
- PROPN: proper noun.
- PUNCT: punctuation.
- SCONJ: subordinating conjunction.
- SYM: symbol.
- VERB: verb.
- X: other.

In [ ]:
# count proportion of Part-of-speech
def pos_analysis(df):

    # Step 1: Create a new DataFrame with selected columns
    pos_df = df[[ 'ADJ', 'VERB', 'PRON', 'NOUN', 'total_pos']].copy()

    # Step 2: Calculate normalized (percentage) columns for each POS type
    pos_df['ADJ%'] = (pos_df['ADJ'] / pos_df['total_pos']) * 100
    pos_df['VERB%'] = (pos_df['VERB'] / pos_df['total_pos']) * 100
    pos_df['PRON%'] = (pos_df['PRON'] / pos_df['total_pos']) * 100
    pos_df['NOUN%'] = (pos_df['NOUN'] / pos_df['total_pos']) * 100

    # Step 3: Calculate the specified POS ratios
    pos_df['Noun_to_Verb'] = pos_df['NOUN'] / (pos_df['VERB'] )
    pos_df['Pron_to_Noun'] = pos_df['PRON'] / (pos_df['NOUN'] )
    pos_df['Noun_to_Adj'] = pos_df['NOUN'] / (pos_df['ADJ'])
    pos_df['Verb_to_Adj'] = pos_df['VERB'] / (pos_df['ADJ'])

    return pos_df

In [ ]:
pos_result = pos_analysis(df_pos)
pos_result

sp_pos,ADJ,VERB,PRON,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
id,,,,,,,,,,,,,
1.667270e+14,4,9,7,11,48,8.333333,18.750000,14.583333,22.916667,1.222222,0.636364,2.750000,2.250000
2.098540e+14,3,4,5,10,27,11.111111,14.814815,18.518519,37.037037,2.500000,0.500000,3.333333,1.333333
2.381550e+14,2,12,11,25,81,2.469136,14.814815,13.580247,30.864198,2.083333,0.440000,12.500000,6.000000
2.481730e+14,5,8,11,28,79,6.329114,10.126582,13.924051,35.443038,3.500000,0.392857,5.600000,1.600000
2.620290e+14,5,10,4,7,43,11.627907,23.255814,9.302326,16.279070,0.700000,0.571429,1.400000,2.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3.032345e+16,0,5,5,15,34,0.000000,14.705882,14.705882,44.117647,3.000000,0.333333,inf,inf
3.052671e+16,0,4,3,2,14,0.000000,28.571429,21.428571,14.285714,0.500000,1.500000,inf,inf
3.071338e+16,4,7,3,12,43,9.302326,16.279070,6.976744,27.906977,1.714286,0.250000,3.000000,1.750000


In [ ]:
pos_result.columns

Index(['ADJ', 'VERB', 'PRON', 'NOUN', 'total_pos', 'ADJ%', 'VERB%', 'PRON%',
       'NOUN%', 'Noun_to_Verb', 'Pron_to_Noun', 'Noun_to_Adj', 'Verb_to_Adj'],
      dtype='object', name='sp_pos')

In [ ]:
# save the result to the orginal csv. file
# pd.merge(): Merges the two DataFrames on the common column 'id'
# how='left': Ensures that all rows from ad_social are kept, even if there’s no matching row in pos_result.

merged_df = pd.merge(df, pos_result[['ADJ', 'VERB', 'PRON', 'NOUN', 'total_pos',
                                            'ADJ%', 'VERB%', 'PRON%','NOUN%',
                                            'Noun_to_Verb', 'Pron_to_Noun', 'Noun_to_Adj', 'Verb_to_Adj']],
                     on='id',
                     how='left')
merged_df

,id,post_owner.username,text,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,...,NOUN,total_pos,ADJ%,VERB%,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj
0,6.147890e+14,SAGEUSA,"🌈🧠 This #MentalHealthAwarenessMonth, SAGE is p...","this , sage is proud to feature our partnersh...",0.000,0.902,0.098,0.8172,Positive,4.0,...,16,69,8.695652,11.594203,7.246377,23.188406,2.000000,0.312500,2.666667,1.333333
1,1.553641e+15,SAGEUSA,💡A study on suicidal behavior and coming out m...,a study on suicidal behavior and coming out mi...,0.149,0.814,0.037,-0.8519,Negative,3.0,...,19,63,1.587302,17.460317,6.349206,30.158730,1.727273,0.210526,19.000000,11.000000
2,1.356778e+15,SAGEUSA,REMINDER‼️🏳️‍🌈 Celebrate Pride Month at SAGE &...,reminder‼‍ celebrate pride month at sage & fri...,0.026,0.643,0.330,0.9609,Positive,3.0,...,14,42,0.000000,21.428571,9.523810,33.333333,1.555556,0.285714,inf,inf
3,1.047445e+15,SAGEUSA,"📣 SAGE thanks Congressman Robert Garcia, Repre...","sage thanks congressman robert garcia, repres...",0.000,0.833,0.167,0.8402,Positive,1.0,...,16,44,2.272727,13.636364,6.818182,36.363636,2.666667,0.187500,16.000000,6.000000
4,1.889815e+15,SAGEUSA,📣 Minnesota is making history! 🏛️💙 SAGE commen...,minnesota is making history! sage commends r...,0.000,0.838,0.162,0.9359,Positive,5.0,...,26,85,7.058824,16.470588,7.058824,30.588235,1.857143,0.230769,4.333333,2.333333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5739,1.069095e+15,SAGEUSA,Check out the link for a variety of informativ...,check out the link for a variety of informativ...,0.000,1.000,0.000,0.0000,Neutral,1.0,...,6,18,5.555556,11.111111,22.222222,33.333333,3.000000,0.666667,6.000000,2.000000
5740,7.236230e+14,SAGEUSA,Thank you to all who participated in our Inter...,thank you to all who participated in our inter...,0.000,0.823,0.177,0.6808,Positive,2.0,...,5,28,3.571429,14.285714,32.142857,17.857143,1.250000,1.800000,5.000000,4.000000
5741,1.206087e+15,SAGEUSA,On The Huffington Post too!,on the huffington post too!,0.000,1.000,0.000,0.0000,Neutral,1.0,...,0,5,0.000000,20.000000,20.000000,0.000000,0.000000,inf,NaN,inf
5742,9.490401e+15,SAGEUSA,Don't forget our SAGE Harlem Thanksgiving Potl...,don't forget our sage harlem thanksgiving potl...,0.000,0.734,0.266,0.3699,Positive,1.0,...,3,9,0.000000,33.333333,11.111111,33.333333,1.000000,0.333333,inf,inf


# SentimentAnalysis using Emotion Dictionary

In [ ]:
Valence_Emotion = pd.read_csv('/content/Valence Emotion dictionary.csv')
Valence_Emotion

,Unnamed: 0,Word,V.Mean.Sum,V.SD.Sum,V.Rat.Sum,A.Mean.Sum,A.SD.Sum,A.Rat.Sum,D.Mean.Sum,D.SD.Sum,...,A.Rat.L,A.Mean.H,A.SD.H,A.Rat.H,D.Mean.L,D.SD.L,D.Rat.L,D.Mean.H,D.SD.H,D.Rat.H
0,0,.,0.00,0.00,0,0.00,0.00,0,0.00,0.00,...,0,0.00,0.00,0,0.00,0.00,0,0.00,0.00,0
1,1,aardvark,6.26,2.21,19,2.41,1.40,22,4.27,1.75,...,11,2.55,1.29,11,4.12,1.64,8,4.43,1.99,7
2,2,abalone,5.30,1.59,20,2.65,1.90,20,4.95,1.79,...,12,2.38,1.92,8,5.55,2.21,11,4.36,1.03,11
3,3,abandon,2.84,1.54,19,3.73,2.43,22,3.32,2.50,...,11,3.82,2.14,11,2.77,2.09,13,4.11,2.93,9
4,4,abandonment,2.63,1.74,19,4.95,2.64,21,2.64,1.81,...,14,5.29,2.63,7,2.31,1.45,16,3.08,2.19,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13911,13911,zone,4.75,2.05,20,3.78,2.53,18,5.23,1.82,...,9,4.89,1.76,9,5.09,1.81,11,5.36,1.91,11
13912,13912,zoning,4.65,1.60,20,3.77,1.95,22,4.47,2.20,...,12,3.70,1.57,10,5.17,2.32,6,4.15,2.15,13
13913,13913,zoo,7.00,1.58,21,5.63,2.54,19,6.33,2.56,...,13,5.17,2.14,6,5.67,2.87,12,7.22,1.86,9
13914,13914,zoom,5.86,1.53,21,5.68,2.54,19,5.90,2.17,...,8,6.27,2.45,11,6.00,2.12,9,5.83,2.29,12


In [ ]:
Valence_Emotion_dict = {}

for i in Valence_Emotion.index:
  Valence_Emotion_dict[Valence_Emotion['Word'][i]] = [Valence_Emotion['V.Mean.Sum'][i],
                                                      Valence_Emotion['A.Mean.Sum'][i],
                                                      Valence_Emotion['D.Mean.Sum'][i]]

#test
Valence_Emotion_dict['abandon']

[np.float64(2.84), np.float64(3.73), np.float64(3.32)]

In [ ]:
SemD = pd.read_csv('/content/SemD.csv')
SemD

,item,mean_cos,SemD,BNC_wordcount,BNC_contexts,BNC_freq,lg_BNC_freq
0,aa,0.020000,1.69,577,314.0,6.6,0.88
1,aah,0.085245,1.07,92,58.0,1.1,0.31
2,aback,0.025864,1.59,294,293.0,3.4,0.64
3,abacus,0.022685,1.64,51,40.0,0.6,0.20
4,abandon,0.008199,2.09,1257,1193.0,14.4,1.19
...,...,...,...,...,...,...,...
31734,zoom,0.051591,1.29,241,161.0,2.8,0.58
31735,zoomed,0.029300,1.53,64,59.0,0.7,0.24
31736,zooming,0.023874,1.62,61,54.0,0.7,0.23
31737,zoos,0.079977,1.10,144,87.0,1.7,0.42


In [ ]:
SemD_dict = {}
for i in SemD.index:
  SemD_dict[SemD['item'][i]] = SemD['SemD'][i]

#test
SemD_dict['weird']

np.float64(1.47)

In [ ]:
# define a function to retrieve values (valuence, arousal, dominance, semd) from the dictionaries**

def get_emotion_values(word):
    valence = Valence_Emotion_dict.get(word, [None, None, None])[0]
    arousal = Valence_Emotion_dict.get(word, [None, None, None])[1]
    dominance = Valence_Emotion_dict.get(word, [None, None, None])[2]
    semd = SemD_dict.get(word, None)
    return pd.Series([valence, arousal, dominance, semd], index=['Valence', 'Arousal', 'Dominance', 'SemD'])

In [ ]:
# Apply the helper function to each row in the DataFrame
df_lex[['Valence', 'Arousal', 'Dominance', 'SemD']] = df_lex['word'].apply(get_emotion_values)

df_lex

,id,clean_text,word,sp_tokenized,sp_n_token,sp_lemma,sp_pos,sp_tag,Valence,Arousal,Dominance,SemD
0,6.147890e+14,"this , sage is proud to feature our partnersh...",this,[this],1,[this],[PRON],[DT],NaN,NaN,NaN,2.37
0,6.147890e+14,"this , sage is proud to feature our partnersh...",,[],0,[],[],[],NaN,NaN,NaN,NaN
0,6.147890e+14,"this , sage is proud to feature our partnersh...",sage,[sage],1,[sage],[NOUN],[NN],5.79,3.33,5.25,1.70
0,6.147890e+14,"this , sage is proud to feature our partnersh...",is,[is],1,[be],[AUX],[VBZ],NaN,NaN,NaN,2.37
0,6.147890e+14,"this , sage is proud to feature our partnersh...",proud,[proud],1,[proud],[ADJ],[JJ],7.00,5.55,7.09,1.92
...,...,...,...,...,...,...,...,...,...,...,...,...
5743,3.994127e+15,"saturday, november 19, 2011 from 8 - 10 pm at ...",mad,[mad],1,[mad],[ADJ],[JJ],2.47,5.59,4.90,1.71
5743,3.994127e+15,"saturday, november 19, 2011 from 8 - 10 pm at ...",stu,[stu],1,[stu],[PROPN],[NNP],NaN,NaN,NaN,NaN
5743,3.994127e+15,"saturday, november 19, 2011 from 8 - 10 pm at ...",media,[media],1,[medium],[NOUN],[NNS],NaN,NaN,NaN,1.57
5743,3.994127e+15,"saturday, november 19, 2011 from 8 - 10 pm at ...",and,[and],1,[and],[CCONJ],[CC],NaN,NaN,NaN,2.37


In [ ]:
# save the results to a new cvs. file
df_lex.to_csv('SAGEUSA_lexicon.csv', index=False)

In [ ]:
# For each participant, calculate the mean (average) for the columns 'Valence', 'Arousal', 'Dominance', and 'SemD'.
emotion = df_lex.groupby(['id']).agg({'Valence': lambda x: np.mean(x),
                                 'Arousal': lambda x: np.mean(x),
                                 'Dominance': lambda x: np.mean(x),
                                  'SemD': lambda x: np.mean(x)}).reset_index()
emotion

,id,Valence,Arousal,Dominance,SemD
0,1.667270e+14,5.398000,4.356000,5.564667,2.097250
1,2.098540e+14,6.593636,3.960000,6.242727,2.057200
2,2.381550e+14,5.934643,3.954286,5.742857,2.081143
3,2.481730e+14,5.971667,4.090000,6.110000,2.027681
4,2.620290e+14,5.701250,4.436250,5.447500,2.057222
...,...,...,...,...,...
5726,3.032345e+16,6.116667,3.901333,6.082000,2.060625
5727,3.052671e+16,5.773333,3.400000,5.643333,2.222727
5728,3.071338e+16,5.337143,4.580000,5.495714,2.059048
5729,3.074114e+16,5.342000,4.025333,5.678000,2.065000


In [ ]:
# pd.merge(): Merges the two DataFrames on the common column 'id'
final = pd.merge(merged_df, emotion[['id','Valence', 'Arousal', 'Dominance', 'SemD']],
                     on='id',
                     how='left')
final

,id,post_owner.username,text,clean_text,VADER_neg,VADER_neu,VADER_pos,VADER_compound,VADER_sentiment,sent_count,...,PRON%,NOUN%,Noun_to_Verb,Pron_to_Noun,Noun_to_Adj,Verb_to_Adj,Valence,Arousal,Dominance,SemD
0,6.147890e+14,SAGEUSA,"🌈🧠 This #MentalHealthAwarenessMonth, SAGE is p...","this , sage is proud to feature our partnersh...",0.000,0.902,0.098,0.8172,Positive,4.0,...,7.246377,23.188406,2.000000,0.312500,2.666667,1.333333,6.023500,4.070000,5.871500,2.036250
1,1.553641e+15,SAGEUSA,💡A study on suicidal behavior and coming out m...,a study on suicidal behavior and coming out mi...,0.149,0.814,0.037,-0.8519,Negative,3.0,...,6.349206,30.158730,1.727273,0.210526,19.000000,11.000000,5.348750,3.798125,5.191250,2.016275
2,1.356778e+15,SAGEUSA,REMINDER‼️🏳️‍🌈 Celebrate Pride Month at SAGE &...,reminder‼‍ celebrate pride month at sage & fri...,0.026,0.643,0.330,0.9609,Positive,3.0,...,9.523810,33.333333,1.555556,0.285714,inf,inf,6.279231,4.258462,5.943846,1.943548
3,1.047445e+15,SAGEUSA,"📣 SAGE thanks Congressman Robert Garcia, Repre...","sage thanks congressman robert garcia, repres...",0.000,0.833,0.167,0.8402,Positive,1.0,...,6.818182,36.363636,2.666667,0.187500,16.000000,6.000000,5.627647,3.675882,5.506471,1.884722
4,1.889815e+15,SAGEUSA,📣 Minnesota is making history! 🏛️💙 SAGE commen...,minnesota is making history! sage commends r...,0.000,0.838,0.162,0.9359,Positive,5.0,...,7.058824,30.588235,1.857143,0.230769,4.333333,2.333333,6.042593,3.834074,5.770000,2.073913
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5739,1.069095e+15,SAGEUSA,Check out the link for a variety of informativ...,check out the link for a variety of informativ...,0.000,1.000,0.000,0.0000,Neutral,1.0,...,22.222222,33.333333,3.000000,0.666667,6.000000,2.000000,6.231667,3.748333,5.733333,2.124706
5740,7.236230e+14,SAGEUSA,Thank you to all who participated in our Inter...,thank you to all who participated in our inter...,0.000,0.823,0.177,0.6808,Positive,2.0,...,32.142857,17.857143,1.250000,1.800000,5.000000,4.000000,6.306667,3.815000,5.813333,2.118462
5741,1.206087e+15,SAGEUSA,On The Huffington Post too!,on the huffington post too!,0.000,1.000,0.000,0.0000,Neutral,1.0,...,20.000000,0.000000,0.000000,inf,NaN,inf,5.050000,3.580000,6.320000,2.250000
5742,9.490401e+15,SAGEUSA,Don't forget our SAGE Harlem Thanksgiving Potl...,don't forget our sage harlem thanksgiving potl...,0.000,0.734,0.266,0.3699,Positive,1.0,...,11.111111,33.333333,1.000000,0.333333,inf,inf,5.725000,4.415000,5.072500,1.714000


In [ ]:
# save the results to a new cvs. file
final.to_csv('/content/SAGEUSA_results.csv', index=False)